# FASE 1: Generar Metadata de Negocio (Datos Sintéticos)

## Objetivo
Generar tabla `smart_claims.silver.claims_business_data` con información de negocio realista:
* policy_no, policy_status (active 85%, expired 10%, suspended 5%)
* coverage_limit (20k-200k USD)
* claim_amount (5k-150k USD)
* reported_severity (Minor 30%, Moderate 40%, Severe 30%)
* incident_type (Collision 50%, Theft 20%, Weather 20%, Vandalism 10%)
* incident_date (últimos 60 días), incident_hour (0-23)
* claim_date, days_to_report

In [0]:
%sql
-- 🔑 PASO CRÍTICO: El owner del catálogo 'smart_claims' debe ejecutar estos comandos
-- para otorgar permisos al usuario empresariox2004@gmail.com
--
-- Estos permisos son necesarios para:
--   1. USE CATALOG: Acceder al catálogo smart_claims
--   2. USE SCHEMA: Acceder a los schemas silver y gold
--   3. SELECT: Leer datos de las tablas existentes
--   4. CREATE TABLE: Crear las tablas silver.claims_business_data y gold.claims_final_insights
--   5. MODIFY: Poder sobrescribir las tablas con mode("overwrite")

-- Otorgar permisos a nivel de catálogo (hereda a todos los schemas)
GRANT USE CATALOG, USE SCHEMA, SELECT, CREATE TABLE, MODIFY ON CATALOG smart_claims TO `empresariox2004@gmail.com`;

-- Verificar permisos otorgados
SHOW GRANTS ON CATALOG smart_claims;

-- 🚨 IMPORTANTE: Una vez ejecutado este comando, empresariox2004@gmail.com podrá:
--   - Leer la tabla smart_claims.gold.claim_damage_predictions
--   - Crear la tabla smart_claims.silver.claims_business_data
--   - Crear la tabla smart_claims.gold.claims_final_insights
--   - Ejecutar todas las queries SQL del notebook

Principal,ActionType,ObjectType,ObjectKey
empresariox2004@gmail.com,SELECT,CATALOG,smart_claims
empresariox2004@gmail.com,CREATE TABLE,CATALOG,smart_claims
empresariox2004@gmail.com,USE CATALOG,CATALOG,smart_claims
empresariox2004@gmail.com,USE SCHEMA,CATALOG,smart_claims
empresariox2004@gmail.com,MODIFY,CATALOG,smart_claims


In [0]:
# Cargar la tabla de predicciones de Gold
df_predictions = spark.table("smart_claims.gold.claim_damage_predictions")

total_claims = df_predictions.count()
print(f"✓ Predicciones cargadas: {total_claims:,} registros")
print(f"\nColumnas: {', '.join(df_predictions.columns)}")

✓ Predicciones cargadas: 13,001 registros

Columnas: image_name, claim_no, chassis_no, predicted_damage_level, prediction_confidence, path, prediction_timestamp


In [0]:
from pyspark.sql.functions import (
    col, lit, rand, floor, when, expr, date_sub, current_date, 
    concat, lpad, datediff, hour, monotonically_increasing_id
)
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType
import random

# Obtener los claim_no únicos
df_base = df_predictions.select("claim_no").distinct()

# Generar datos sintéticos
df_business = df_base \
    .withColumn("_rand", rand(seed=42)) \
    .withColumn("_rand2", rand(seed=123)) \
    .withColumn("_rand3", rand(seed=456)) \
    .withColumn("_rand4", rand(seed=789)) \
    .withColumn("_rand5", rand(seed=999)) \
    .withColumn("_rand6", rand(seed=111)) \
    .withColumn("_rand7", rand(seed=222)) \
    .withColumn("_id", monotonically_increasing_id()) \
    \
    .withColumn("policy_no", 
                concat(lit("POL-"), lpad((col("_id") % 10000).cast("string"), 6, "0"))) \
    \
    .withColumn("policy_status",
                when(col("_rand") < 0.85, lit("active"))
                .when(col("_rand") < 0.95, lit("expired"))
                .otherwise(lit("suspended"))) \
    \
    .withColumn("coverage_limit",
                (20000 + floor(col("_rand2") * 180000)).cast("double")) \
    \
    .withColumn("claim_amount",
                (5000 + floor(col("_rand3") * 145000)).cast("double")) \
    \
    .withColumn("reported_severity",
                when(col("_rand4") < 0.30, lit("Minor"))
                .when(col("_rand4") < 0.70, lit("Moderate"))
                .otherwise(lit("Severe"))) \
    \
    .withColumn("incident_type",
                when(col("_rand5") < 0.50, lit("Collision"))
                .when(col("_rand5") < 0.70, lit("Theft"))
                .when(col("_rand5") < 0.90, lit("Weather"))
                .otherwise(lit("Vandalism"))) \
    \
    .withColumn("days_back", floor(col("_rand6") * 60).cast("int")) \
    .withColumn("incident_date", date_sub(current_date(), col("days_back"))) \
    \
    .withColumn("incident_hour", floor(col("_rand7") * 24).cast("int")) \
    \
    .withColumn("days_to_report", floor(col("_rand2") * 15).cast("int")) \
    .withColumn("claim_date", expr("date_add(incident_date, days_to_report)")) \
    \
    .select(
        "claim_no",
        "policy_no",
        "policy_status",
        "coverage_limit",
        "claim_amount",
        "reported_severity",
        "incident_type",
        "incident_date",
        "incident_hour",
        "claim_date",
        "days_to_report"
    )

print(f"✓ Datos sintéticos generados: {df_business.count():,} registros")
print("\nMuestra de datos:")
display(df_business.limit(5))

✓ Datos sintéticos generados: 13,001 registros

Muestra de datos:


claim_no,policy_no,policy_status,coverage_limit,claim_amount,reported_severity,incident_type,incident_date,incident_hour,claim_date,days_to_report
891ca2c0-68a2-423c-a8c0-e69fa34f522d,POL-000000,active,90792.0,116048.0,Minor,Weather,2026-03-25,13,2026-03-30,5
172c0cc8-d9b9-4930-8870-929547dd86bd,POL-000001,active,160529.0,148431.0,Moderate,Collision,2026-04-03,22,2026-04-14,11
f2626bbf-b54d-476f-bb0d-33126abcb4cb,POL-000002,active,47611.0,27801.0,Minor,Collision,2026-03-24,19,2026-03-26,2
410f675c-e490-4bc3-8523-8acd83ebf9de,POL-000003,active,182410.0,80985.0,Severe,Vandalism,2026-04-24,23,2026-05-07,13
139acbf7-5df4-4a2f-9560-63fe35c6604d,POL-000004,active,103341.0,24807.0,Moderate,Collision,2026-04-21,18,2026-04-27,6


In [0]:
# Guardar en Silver
df_business.write.mode("overwrite").saveAsTable("smart_claims.silver.claims_business_data")

print("✓ Tabla smart_claims.silver.claims_business_data creada")
print(f"  Registros: {df_business.count():,}")
print(f"\nDistribución de policy_status:")
display(df_business.groupBy("policy_status").count().orderBy("policy_status"))

print("\nDistribución de reported_severity:")
display(df_business.groupBy("reported_severity").count().orderBy("reported_severity"))

print("\nDistribución de incident_type:")
display(df_business.groupBy("incident_type").count().orderBy("incident_type"))

✓ Tabla smart_claims.silver.claims_business_data creada
  Registros: 13,001

Distribución de policy_status:


policy_status,count
active,10986
expired,1349
suspended,666



Distribución de reported_severity:


reported_severity,count
Minor,3903
Moderate,5211
Severe,3887



Distribución de incident_type:


incident_type,count
Collision,6491
Theft,2609
Vandalism,1312
Weather,2589


# FASE 2: Aplicar Reglas de Negocio

## Objetivo
Unir predicciones con metadata de negocio y aplicar 5 reglas:
1. **Póliza inactiva**: policy_status != 'active' → INVESTIGATE
2. **Monto excede cobertura**: claim_amount > coverage_limit * 1.1 → INVESTIGATE
3. **Reporte tardío**: days_to_report > 7 → INVESTIGATE
4. **Discrepancia severidad**: abs(predicted_level - reported_level) >= 2 → INVESTIGATE
5. **Baja confianza**: prediction_confidence < 0.60 → INVESTIGATE

Decisión: **APPROVED** si pasa todas, **INVESTIGATE** si falla alguna.

In [0]:
from pyspark.sql.functions import col

# Cargar ambas tablas
df_predictions = spark.table("smart_claims.gold.claim_damage_predictions")
df_business = spark.table("smart_claims.silver.claims_business_data")

# Unir por claim_no
df_joined = df_predictions.join(df_business, on="claim_no", how="inner")

print(f"✓ Tablas unidas: {df_joined.count():,} registros")
print(f"\nColumnas totales: {len(df_joined.columns)}")
print(f"Columnas: {', '.join(df_joined.columns)}")

✓ Tablas unidas: 13,001 registros

Columnas totales: 17
Columnas: claim_no, image_name, chassis_no, predicted_damage_level, prediction_confidence, path, prediction_timestamp, policy_no, policy_status, coverage_limit, claim_amount, reported_severity, incident_type, incident_date, incident_hour, claim_date, days_to_report


In [0]:
from pyspark.sql.functions import (
    col, when, lit, abs as spark_abs, array, concat_ws, coalesce, current_timestamp
)

# Mapear niveles de severidad a números para comparación
# predicted: ok=0, minor=1, major=2
# reported: Minor=1, Moderate=2, Severe=3

df_with_rules = df_joined \
    .withColumn("predicted_level",
                when(col("predicted_damage_level") == "ok", lit(0))
                .when(col("predicted_damage_level") == "minor", lit(1))
                .when(col("predicted_damage_level") == "major", lit(2))
                .otherwise(lit(1))) \
    \
    .withColumn("reported_level",
                when(col("reported_severity") == "Minor", lit(1))
                .when(col("reported_severity") == "Moderate", lit(2))
                .when(col("reported_severity") == "Severe", lit(3))
                .otherwise(lit(2))) \
    \
    .withColumn("rule1_failed", 
                when(col("policy_status") != "active", lit(True)).otherwise(lit(False))) \
    .withColumn("rule1_reason",
                when(col("rule1_failed"), lit("Póliza inactiva")).otherwise(lit(None))) \
    \
    .withColumn("rule2_failed",
                when(col("claim_amount") > col("coverage_limit") * 1.1, lit(True)).otherwise(lit(False))) \
    .withColumn("rule2_reason",
                when(col("rule2_failed"), lit("Monto excede cobertura")).otherwise(lit(None))) \
    \
    .withColumn("rule3_failed",
                when(col("days_to_report") > 7, lit(True)).otherwise(lit(False))) \
    .withColumn("rule3_reason",
                when(col("rule3_failed"), lit("Reporte tardío")).otherwise(lit(None))) \
    \
    .withColumn("rule4_failed",
                when(spark_abs(col("predicted_level") - col("reported_level")) >= 2, lit(True)).otherwise(lit(False))) \
    .withColumn("rule4_reason",
                when(col("rule4_failed"), lit("Discrepancia predicción")).otherwise(lit(None))) \
    \
    .withColumn("rule5_failed",
                when(col("prediction_confidence") < 0.60, lit(True)).otherwise(lit(False))) \
    .withColumn("rule5_reason",
                when(col("rule5_failed"), lit("Baja confianza")).otherwise(lit(None))) \
    \
    .withColumn("any_rule_failed",
                col("rule1_failed") | col("rule2_failed") | col("rule3_failed") | 
                col("rule4_failed") | col("rule5_failed")) \
    \
    .withColumn("decision_status",
                when(col("any_rule_failed"), lit("INVESTIGATE")).otherwise(lit("APPROVED"))) \
    \
    .withColumn("review_reason",
                concat_ws(", ", 
                         coalesce(col("rule1_reason"), lit("")),
                         coalesce(col("rule2_reason"), lit("")),
                         coalesce(col("rule3_reason"), lit("")),
                         coalesce(col("rule4_reason"), lit("")),
                         coalesce(col("rule5_reason"), lit("")))) \
    \
    .withColumn("review_reason",
                when(col("review_reason") == "", lit(None)).otherwise(col("review_reason"))) \
    \
    .withColumn("severity_match",
                spark_abs(col("predicted_level") - col("reported_level")) < 2)

print("✓ Reglas de negocio aplicadas")
print(f"\nDecisiones:")
display(df_with_rules.groupBy("decision_status").count().orderBy("decision_status"))

print("\nTop 5 motivos de revisión:")
display(df_with_rules.filter(col("decision_status") == "INVESTIGATE")
        .groupBy("review_reason").count()
        .orderBy(col("count").desc())
        .limit(5))

✓ Reglas de negocio aplicadas

Decisiones:


decision_status,count
INVESTIGATE,13001



Top 5 motivos de revisión:


review_reason,count
", , Reporte tardío, , Baja confianza",2865
", , Reporte tardío, Discrepancia predicción, Baja confianza",2204
", Monto excede cobertura, , , Baja confianza",1714
", , , , Baja confianza",1575
", , , Discrepancia predicción, Baja confianza",1286


In [0]:
from pyspark.sql.functions import current_timestamp

# Seleccionar columnas finales para la tabla de insights
df_final_insights = df_with_rules.select(
    # Identificadores
    "claim_no",
    "chassis_no",
    "policy_no",
    
    # Información de negocio
    "incident_type",
    "incident_date",
    "incident_hour",
    "claim_amount",
    "coverage_limit",
    
    # Predicción del modelo
    "predicted_damage_level",
    "prediction_confidence",
    
    # Severidad reportada
    "reported_severity",
    
    # Validación
    "policy_status",
    "days_to_report",
    
    # Decisión final
    "decision_status",
    "review_reason",
    "severity_match",
    
    # Metadata
    "prediction_timestamp"
)

# Guardar en Gold
df_final_insights.write.mode("overwrite").saveAsTable("smart_claims.gold.claims_final_insights")

print("✓ Tabla smart_claims.gold.claims_final_insights creada")
print(f"  Registros: {df_final_insights.count():,}")
print(f"  Columnas: {len(df_final_insights.columns)}")

print("\nMuestra de la tabla final:")
display(df_final_insights.limit(10))

print("\nEstadísticas finales:")
total = df_final_insights.count()
approved = df_final_insights.filter(col("decision_status") == "APPROVED").count()
investigate = df_final_insights.filter(col("decision_status") == "INVESTIGATE").count()
matches = df_final_insights.filter(col("severity_match") == True).count()

print(f"  Total: {total:,}")
print(f"  Aprobados: {approved:,} ({approved/total*100:.1f}%)")
print(f"  A investigar: {investigate:,} ({investigate/total*100:.1f}%)")
print(f"  Coincidencias severidad: {matches:,} ({matches/total*100:.1f}%)")

print("\n" + "="*70)
print("✅ FASES 1 y 2 COMPLETADAS")
print("="*70)

✓ Tabla smart_claims.gold.claims_final_insights creada
  Registros: 13,001
  Columnas: 17

Muestra de la tabla final:


claim_no,chassis_no,policy_no,incident_type,incident_date,incident_hour,claim_amount,coverage_limit,predicted_damage_level,prediction_confidence,reported_severity,policy_status,days_to_report,decision_status,review_reason,severity_match,prediction_timestamp
83157c34-9212-4946-8bc4-64b799e47dbb,LSJA24U60LS012452,POL-000000,Weather,2026-03-25,13,116048.0,90792.0,major,0.35070324,Minor,active,5,INVESTIGATE,", Monto excede cobertura, , , Baja confianza",true,2026-04-24T01:44:48.123Z
d56437f5-c21b-48bf-b5e9-94976b9090d9,ZAREAEAN2K7613358,POL-000001,Collision,2026-04-03,22,148431.0,160529.0,ok,0.35070387,Moderate,active,11,INVESTIGATE,", , Reporte tardío, Discrepancia predicción, Baja confianza",false,2026-04-24T01:44:48.123Z
255ba640-9b63-436c-b810-43e39660e7a1,RKLBB9HE1F5054283,POL-000002,Collision,2026-03-24,19,27801.0,47611.0,major,0.3417953,Minor,active,2,INVESTIGATE,", , , , Baja confianza",true,2026-04-24T01:44:48.123Z
6f1721bc-3325-40e8-a513-61d6d9f3cbaa,JMYJNP13VCA700948,POL-000003,Vandalism,2026-04-24,23,80985.0,182410.0,major,0.35142958,Severe,active,13,INVESTIGATE,", , Reporte tardío, , Baja confianza",true,2026-04-24T01:44:48.123Z
c9f8c1e8-3fa7-465f-8af1-24f26f4a2a9d,MDHAK3CR9FG504346,POL-000004,Collision,2026-04-21,18,24807.0,103341.0,ok,0.3537034,Moderate,active,6,INVESTIGATE,", , , Discrepancia predicción, Baja confianza",false,2026-04-24T01:44:48.123Z
5275d1ad-0220-413b-8d1f-58fa04450698,JL7B6E1P7FK021007,POL-000005,Theft,2026-03-05,22,66844.0,80403.0,major,0.3417953,Minor,active,5,INVESTIGATE,", , , , Baja confianza",true,2026-04-24T01:44:48.123Z
d72b6362-63ea-46e2-8023-82a89b475ddc,KMHJT81CXAU028705,POL-000006,Weather,2026-03-08,10,109018.0,177299.0,ok,0.342183,Severe,active,13,INVESTIGATE,", , Reporte tardío, Discrepancia predicción, Baja confianza",false,2026-04-24T01:44:48.123Z
bebc3f05-9556-4c67-a9c5-f57406e689a0,JTMHU09J7D4069690,POL-000007,Theft,2026-04-22,10,103549.0,45586.0,minor,0.35383958,Severe,active,2,INVESTIGATE,", Monto excede cobertura, , Discrepancia predicción, Baja confianza",false,2026-04-24T01:44:48.123Z
74742328-05d4-4ab2-b8c0-1efb2e4e33fd,KMHSU81EXEU231635,POL-000008,Weather,2026-04-12,10,24370.0,197639.0,ok,0.35070387,Moderate,expired,14,INVESTIGATE,"Póliza inactiva, , Reporte tardío, Discrepancia predicción, Baja confianza",false,2026-04-24T01:44:48.123Z
f0c011dd-eeff-41db-bec7-b552df8d9bbc,JTDKW9236B5158674,POL-000009,Collision,2026-03-23,18,39551.0,38059.0,minor,0.3414177,Moderate,active,1,INVESTIGATE,", , , , Baja confianza",true,2026-04-24T01:44:48.123Z



Estadísticas finales:
  Total: 13,001
  Aprobados: 0 (0.0%)
  A investigar: 13,001 (100.0%)
  Coincidencias severidad: 7,335 (56.4%)

✅ FASES 1 y 2 COMPLETADAS


# FASE 3: Queries SQL para Análisis y BI

## Objetivo
Crear 8 queries para los 3 bloques del dashboard:

### **Bloque A: Vista de Negocio**
* Q1: Reclamos por severidad reportada
* Q2: Pérdida total por tipo de incidente
* Q3: Frecuencia de reclamos por hora

### **Bloque B: Vista de Predicción**
* Q4: Distribución de predicciones del modelo
* Q5: Comparación severidad reportada vs predicha
* Q6: Coincidencia vs discrepancia

### **Bloque C: Vista de Decisión Final**
* Q7: Casos aprobados vs investigados
* Q8: Top motivos de revisión

In [0]:
%sql
-- Q1: Reclamos por severidad reportada
-- Bloque A: Vista de Negocio

SELECT 
  reported_severity,
  COUNT(*) as total_claims,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM smart_claims.gold.claims_final_insights
GROUP BY reported_severity
ORDER BY total_claims DESC

reported_severity,total_claims,percentage
Moderate,5211,40.08
Minor,3903,30.02
Severe,3887,29.90


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q2: Pérdida total por tipo de incidente
-- Bloque A: Vista de Negocio

SELECT 
  incident_type,
  COUNT(*) as total_claims,
  ROUND(SUM(claim_amount), 2) as total_loss,
  ROUND(AVG(claim_amount), 2) as avg_claim_amount
FROM smart_claims.gold.claims_final_insights
GROUP BY incident_type
ORDER BY total_loss DESC

incident_type,total_claims,total_loss,avg_claim_amount
Collision,6491,4.97214463E8,76600.6
Theft,2609,2.00965118E8,77027.64
Weather,2589,1.978526E8,76420.47
Vandalism,1312,1.02108835E8,77826.86


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q3: Frecuencia de reclamos por hora del incidente
-- Bloque A: Vista de Negocio

SELECT 
  incident_hour,
  COUNT(*) as claim_count
FROM smart_claims.gold.claims_final_insights
GROUP BY incident_hour
ORDER BY incident_hour

incident_hour,claim_count
0,546
1,554
2,545
3,570
4,543
5,553
6,540
7,513
8,514
9,526


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q4: Distribución de predicciones del modelo
-- Bloque B: Vista de Predicción

SELECT 
  predicted_damage_level,
  COUNT(*) as prediction_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage,
  ROUND(AVG(prediction_confidence), 4) as avg_confidence
FROM smart_claims.gold.claims_final_insights
GROUP BY predicted_damage_level
ORDER BY prediction_count DESC

predicted_damage_level,prediction_count,percentage,avg_confidence
ok,7728,59.44,0.3495
major,4457,34.28,0.3416
minor,816,6.28,0.3468


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q5: Comparación entre severidad reportada y predicha
-- Bloque B: Vista de Predicción

SELECT 
  reported_severity,
  predicted_damage_level,
  COUNT(*) as count
FROM smart_claims.gold.claims_final_insights
GROUP BY reported_severity, predicted_damage_level
ORDER BY reported_severity, predicted_damage_level

reported_severity,predicted_damage_level,count
Minor,major,1366
Minor,minor,246
Minor,ok,2291
Moderate,major,1761
Moderate,minor,341
Moderate,ok,3109
Severe,major,1330
Severe,minor,229
Severe,ok,2328


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q6: Coincidencia vs discrepancia entre predicción y reporte
-- Bloque B: Vista de Predicción

SELECT 
  CASE 
    WHEN severity_match = true THEN 'Coincidencia'
    ELSE 'Discrepancia'
  END as match_status,
  COUNT(*) as count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM smart_claims.gold.claims_final_insights
GROUP BY severity_match
ORDER BY count DESC

match_status,count,percentage
Coincidencia,7335,56.42
Discrepancia,5666,43.58


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q7: Casos aprobados vs investigados
-- Bloque C: Vista de Decisión Final

SELECT 
  decision_status,
  COUNT(*) as count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage,
  ROUND(AVG(claim_amount), 2) as avg_claim_amount
FROM smart_claims.gold.claims_final_insights
GROUP BY decision_status
ORDER BY count DESC

decision_status,count,percentage,avg_claim_amount
INVESTIGATE,13001,100.00,76774.17


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q9: Predicción vs decisión final
-- Bloque C: Vista de Decisión Final
-- Muestra cómo las predicciones del modelo se traducen en decisiones operativas

SELECT 
  predicted_damage_level,
  decision_status,
  COUNT(*) as count,
  ROUND(AVG(prediction_confidence), 4) as avg_confidence,
  ROUND(AVG(claim_amount), 2) as avg_claim_amount
FROM smart_claims.gold.claims_final_insights
GROUP BY predicted_damage_level, decision_status
ORDER BY predicted_damage_level, decision_status

predicted_damage_level,decision_status,count,avg_confidence,avg_claim_amount
major,INVESTIGATE,4457,0.3416,77258.01
minor,INVESTIGATE,816,0.3468,76806.94
ok,INVESTIGATE,7728,0.3495,76491.67


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Q8: Top motivos de revisión (reglas que fallaron)
-- Bloque C: Vista de Decisión Final

SELECT 
  review_reason,
  COUNT(*) as count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM smart_claims.gold.claims_final_insights WHERE decision_status = 'INVESTIGATE'), 2) as percentage_of_investigated
FROM smart_claims.gold.claims_final_insights
WHERE decision_status = 'INVESTIGATE'
  AND review_reason IS NOT NULL
GROUP BY review_reason
ORDER BY count DESC
LIMIT 10

review_reason,count,percentage_of_investigated
", , Reporte tardío, , Baja confianza",2865,22.04
", , Reporte tardío, Discrepancia predicción, Baja confianza",2204,16.95
", Monto excede cobertura, , , Baja confianza",1714,13.18
", , , , Baja confianza",1575,12.11
", , , Discrepancia predicción, Baja confianza",1286,9.89
", Monto excede cobertura, , Discrepancia predicción, Baja confianza",1257,9.67
"Póliza inactiva, , Reporte tardío, , Baja confianza",524,4.03
"Póliza inactiva, , Reporte tardío, Discrepancia predicción, Baja confianza",406,3.12
"Póliza inactiva, , , , Baja confianza",314,2.42
"Póliza inactiva, Monto excede cobertura, , , Baja confianza",274,2.11


Databricks visualization. Run in Databricks to view.